<a href="https://colab.research.google.com/github/sabithamanoj/customer-churn-ml-deployment/blob/main/src/Readme_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flask API Deployment for Customer Churn Prediction

This folder contains the deployment implementation of the trained Customer Churn Prediction model.

The trained machine learning model was exported using `joblib` and integrated into a Flask REST API. The API accepts customer details as JSON input, performs the required preprocessing, generates predictions, and returns churn probability.

---

# Deployment Workflow

```
Customer Input (JSON)
        |
        ▼
     Flask API
        |
        ▼
 Input Preprocessing
        |
        ▼
 Feature Alignment
        |
        ▼
 Trained ML Model
        |
        ▼
 Prediction Response
```

---

# Folder Structure

```
src/

├── app.py
│   Flask application containing API endpoints.

├── preprocess.py
│   Handles preprocessing of incoming customer data
│   and converts it into the format expected by the model.

├── predict.py
│   Loads the trained model and generates predictions.

├── test_model_loading.py
│   Verifies that the trained model and artifacts
│   can be loaded successfully.

└── test_prediction.py
    Tests the complete prediction pipeline locally.


api_tests/

└── customer_churn_postman_collection.json
    Postman collection for testing API endpoints.
```

---

# 1. Model Loading (`predict.py`)

The prediction module performs the following tasks:

- Loads the trained machine learning model:

```
models/churn_model.pkl
```

- Loads feature information:

```
models/feature_names.pkl
```

- Loads preprocessing artifacts:

```
models/scaler.pkl
```

- Ensures incoming data follows the same feature order used during training.

The model expects exactly the same 30 features that were generated during training.

---

# 2. Input Preprocessing (`preprocess.py`)

The preprocessing pipeline converts raw customer information into model-ready input.

Steps performed:

### Convert JSON input to DataFrame

Example:

```python
{
 "gender": "Female",
 "tenure": 12
}
```

is converted into a pandas DataFrame.

---

### Convert TotalCharges

The same conversion used during training is applied:

```python
pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)
```

This ensures numerical consistency.

---

### One-Hot Encoding

Categorical variables are converted into numerical features:

Example:

```
Contract = Month-to-month
```

becomes:

```
Contract_Month-to-month = 1
```

---

### Feature Alignment

The processed input is matched with the training feature list:

- Missing columns are added with value `0`
- Extra columns are removed
- Column order is preserved

This ensures the deployed model receives exactly the same input format as during training.

---

# 3. Flask API (`app.py`)

The Flask application provides REST endpoints.

## Health Check Endpoint

### Request

```
GET http://127.0.0.1:5000/
```

### Response

```json
{
    "message": "Customer Churn Prediction API is running."
}
```

---

## Prediction Endpoint

### Request

```
POST http://127.0.0.1:5000/predict
```

### Headers

```
Content-Type: application/json
```

### Example Input

```json
{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "InternetService": "Fiber optic",
    "Contract": "Month-to-month",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 80,
    "TotalCharges": 960
}
```

### Response

```json
{
    "prediction": "No Churn",
    "churn_probability": 0.2095
}
```

---

# 4. Local Testing

## Test Model Loading

Run:

```bash
python src/test_model_loading.py
```

Purpose:

- Verify model artifacts are available
- Confirm compatibility between saved model and local environment

---

## Test Prediction Pipeline

Run:

```bash
python src/test_prediction.py
```

Example output:

```
Processed input shape: (1, 30)

Prediction result:
{
 'prediction': 'No Churn',
 'churn_probability': 0.2095
}
```

---

# 5. API Testing Using Postman

A Postman collection is provided:

```
api_tests/customer_churn_postman_collection.json
```

The collection contains:

- Health check request
- Customer churn prediction request

---

## Import Postman Collection

Steps:

1. Open Postman

2. Select:

```
Import
```

3. Choose:

```
api_tests/customer_churn_postman_collection.json
```

4. Start Flask API:

```bash
python src/app.py
```

5. Execute requests from Postman.

---

# Postman Prediction Test

Request:

```
POST http://127.0.0.1:5000/predict
```

Body:

```json
{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12,
    "PhoneService": "Yes",
    "InternetService": "Fiber optic",
    "Contract": "Month-to-month",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 80,
    "TotalCharges": 960
}
```

Response:

```json
{
    "prediction": "No Churn",
    "churn_probability": 0.2095
}
```

---

# Logging

The Flask API records important events:

- Application startup
- Incoming requests
- Preprocessing completion
- Prediction results
- Errors

Example:

```
INFO - Received input
INFO - Preprocessing completed. Shape: (1,30)
INFO - Prediction result generated
```

---

# Deployment Status

Completed:

✅ Model loading  
✅ Input preprocessing pipeline  
✅ Flask REST API  
✅ Prediction endpoint  
✅ API health check  
✅ Logging  
✅ Local API testing using Postman  

